# StormScope-EU nowcasting on MTG-I1 FCI -> Arraylake

Runs [`StormScopeMeteosatEU`](https://nvidia.github.io/earth2studio/main/modules/generated/models/px/StormScopeMeteosatEU/)
— NVIDIA's generative diffusion nowcasting model for Meteosat Third Generation FCI imagery —
end to end: fetch six consecutive FCI frames from the EUMETSAT Data Store, roll the model
forward autoregressively in 10-minute steps, **write each step to Icechunk on Arraylake as it
is produced**, and plot a true-colour RGB composite.

Based on the Earth2Studio example
[`04_stormscope_meteosat_example`](https://nvidia.github.io/earth2studio/main/examples/04_nowcasting/04_stormscope_meteosat_example/)
and the step-by-step [StormScope-EU run workflow](https://hackmd.io/@ukEyv-C_TnK1RVN9gH211w/BkSIAjzOGl).

**This needs a CUDA GPU.** The sections below run it on a Coiled GPU VM in the
`earthmover-devs` workspace.

## Run this on Coiled

Background: [Running Jupyter notebooks on a GPU with Coiled](https://docs.coiled.io/blog/jupyter-notebook-gpu.html).
Coiled installs the NVIDIA driver and CUDA runtime on the VM for you; you only supply the
Python environment.

All commands below assume credentials are loaded from `.env` first:

```bash
set -a; source .env; set +a     # EUMETSAT_CONSUMER_KEY / _SECRET, ARRAYLAKE_API_TOKEN
```

### 1. Build the software environment (once per region)

Already built as `earthmover-devs/stormscope-eu`. The definition lives in this repo under
`coiled/`, so rebuild it from there:

```bash
uv tool install coiled          # or: pip install "coiled[notebook]"
coiled login                    # if not already configured

coiled env create \
  --name stormscope-eu \
  --workspace earthmover-devs \
  --gpu-enabled \
  --architecture x86_64 \
  --conda coiled/environment-stormscope-eu.yml \
  --pip coiled/requirements-stormscope-eu.txt
```

**Images are built per region.** Launching in a region the env has never run in silently
queues a fresh ~9.6 GB build, and the VM gives up waiting for it:

```
Cluster status is error (reason: Scheduler Stopped -> Timed out waiting for process to phone home)
```

That is not a code error — `coiled logs <cluster-id>` for such a failure is nothing but
`Software environment has not finished building yet` on repeat. The build keeps going after
the VM dies, so wait for it and relaunch:

```bash
coiled env list --workspace earthmover-devs | grep stormscope-eu   # queued -> building -> built
```

Inspect or force a rebuild later:

```bash
coiled env inspect earthmover-devs/stormscope-eu
coiled env create ... --force-rebuild        # after editing the requirements file
```

### 2. Launch a GPU notebook session

> The subcommand is `coiled notebook start` — there is no `coiled notebook run`.
> `coiled run` is the batch/script equivalent (section 3).

```bash
coiled notebook start \
  --workspace earthmover-devs \
  --software stormscope-eu \
  --name stormscope-eu \
  --gpu \
  --vm-type g6e.xlarge --vm-type g6e.2xlarge \
  --vm-type g5.2xlarge --vm-type g5.4xlarge \
  --vm-type g6.2xlarge --vm-type g6.4xlarge \
  --region us-east-1 \
  --disk-size 200 \
  --idle-timeout '2 hours' \
  --sync \
  --allow-sync-from me \
  --env EUMETSAT_CONSUMER_KEY=$EUMETSAT_CONSUMER_KEY \
  --env EUMETSAT_CONSUMER_SECRET=$EUMETSAT_CONSUMER_SECRET \
  --env ARRAYLAKE_API_TOKEN=$ARRAYLAKE_API_TOKEN
```

JupyterLab opens in your browser. Re-running the *same* command with the same `--name`
re-opens the existing session rather than starting a second VM — that is also how you turn
`--sync` on for a session that was launched without it.

`--sync` mirrors this working directory onto the VM (needs [mutagen](https://mutagen.io/)),
so this notebook and anything it writes to `outputs/` come back to your laptop.

| flag | why |
| --- | --- |
| `--gpu` | on its own this gives Coiled's default `g4dn.xlarge` (Tesla T4, 16 GB) — too small here, so always pair it with `--vm-type`. |
| `--vm-type` (x6) | Coiled takes the first type with capacity. **Pass several.** GPU capacity is genuinely scarce: a lone `--vm-type g6e.xlarge` failed across all four `us-west-2` AZs with `insufficient capacity`. Listed largest first — `g6e` is 1x L40S (48 GB), `g5`/`g6` are A10G/L4 (24 GB) and are fine at `ensemble_size=1`. `p5.48xlarge` gets you the H100 the upstream example used. |
| `--region us-east-1` | same region as the Arraylake repo's bucket, so the Icechunk writes don't cross regions. Changing it means a per-region image rebuild — see section 1. |
| `--disk-size 200` | the env is ~9.6 GB, plus the StormScope checkpoint and cached FCI granules. |
| `--idle-timeout` | the VM shuts itself down; default is 4 hours. |
| `--env ...` | EUMETSAT and Arraylake credentials, transmitted securely (see below). |

Stop it when you're done — Coiled bills for the VM while it's up:

```bash
coiled notebook stop --name stormscope-eu --workspace earthmover-devs
```

### 3. Or run it headless as a script

[`06_stormscope_eu_pipeline.py`](06_stormscope_eu_pipeline.py) is the headless twin of this
notebook: same schema and compute path, as a CLI. Note it publishes to the root group of
`gtc-berlin/stormscope-eu_gtc_berlin_2026`, the same repo and group as this notebook. Runs
from either accumulate in one cube: each appends its init to `time`.

```bash
coiled run \
  --workspace earthmover-devs \
  --software stormscope-eu \
  --name stormscope-eu-run \
  --gpu \
  --vm-type g6e.xlarge --vm-type g6e.2xlarge \
  --vm-type g5.2xlarge --vm-type g5.4xlarge \
  --vm-type g6.2xlarge --vm-type g6.4xlarge \
  --region us-east-1 \
  --disk-size 200 \
  --keepalive 1h \
  --env EUMETSAT_CONSUMER_KEY=$EUMETSAT_CONSUMER_KEY \
  --env EUMETSAT_CONSUMER_SECRET=$EUMETSAT_CONSUMER_SECRET \
  --env ARRAYLAKE_API_TOKEN=$ARRAYLAKE_API_TOKEN \
  python 06_stormscope_eu_pipeline.py --publish-target arraylake --arraylake-repo gtc-berlin/stormscope-eu_gtc_berlin_2026
```

**`--keepalive` matters.** By default the VM shuts down the instant the command exits, which
takes any shell you had open on it with it. Add `--detach` to return the prompt immediately
and follow the run with `coiled logs` instead of staying attached.

Killing a run mid-rollout is safe but partial: each 10-minute step is its own Icechunk commit,
so the repo keeps whatever steps had committed. Re-running the same init reuses its `time` slice
and overwrites it step by step; a new init is appended. Nothing is ever wiped.

**Do not add `--sync` to `coiled run`.** `coiled run` uploads the script by itself, and the
mutagen session `--sync` starts is fragile: editing anything in this directory while the run
is in flight can fail it with `unable to flush session: session is not currently able to
synchronize`, which terminates the remote process mid-rollout. That is exactly how the
2026-09-08 run died after committing only the initial condition — not a model or Icechunk
error. `--sync` stays right for `coiled notebook start` in section 2, where an interactive
session is the whole point. The cost of dropping it is that nothing is copied back, so
`outputs/` exists only on the VM; read results from Arraylake instead.

**Coiled logs carry your secrets.** On a bad exit Coiled prints the full
`docker exec ... --env ...` command line, which contains the literal values of
`EUMETSAT_CONSUMER_KEY`, `EUMETSAT_CONSUMER_SECRET` and `ARRAYLAKE_API_TOKEN`. Do not paste that
output into Slack or an issue, and rotate the credentials if you already have.

### 4. Getting into a running VM

```bash
coiled cluster list --workspace earthmover-devs | head        # names, ids, status

coiled cluster ssh <cluster-id> --workspace earthmover-devs --dask   # software container
coiled cluster ssh <cluster-id> --workspace earthmover-devs          # host VM
coiled logs <cluster-id> --workspace earthmover-devs                 # scheduler logs
```

`--dask` puts you in the **software container**, where `python` is the `stormscope-eu` env and
the synced working directory lives at `/scratch/synced` — that is where the script runs and
`outputs/` is written. Without it you get the **host VM**, which is the level for `nvidia-smi`,
`docker` and `df`. SSH needs no setup: `coiled run` defaults to `--allow-ssh-from me`.

Because the run above set `--name stormscope-eu-run`, further `coiled run` invocations with
that name land on the *same* VM instead of starting a new one:

```bash
coiled run --name stormscope-eu-run --workspace earthmover-devs nvidia-smi
coiled run --name stormscope-eu-run --workspace earthmover-devs -it python
```

## Credentials

**EUMETSAT Data Store** (model input)

1. Register (free) at the EUMETSAT User Portal: <https://user.eumetsat.int>
2. Generate a Consumer Key / Consumer Secret pair at <https://api.eumetsat.int/api-key/>
3. Expose them as `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET`, either via
   `coiled notebook start --env ...` above, a local `.env` file (loaded below by
   `python-dotenv`), or `eumdac set-credentials <key> <secret>` which writes
   `~/.eumdac/credentials`. `MeteosatFCI` reads either.

**Arraylake** (model output)

Set `ARRAYLAKE_API_TOKEN` to an `ema_...` API token, or run `al auth login` once to cache
credentials in `~/.config/arraylake/`. The backend below passes the token to `Client` explicitly
(the same variable `06_stormscope_eu_pipeline.py` reads), so `ARRAYLAKE_API_TOKEN` must be set.
On a Coiled VM there is no browser, so pass the token via `--env`.

## What's in the environment

Two things `pip install earth2studio[stormscope,data]` will not get right on a CPU build
machine, and how `coiled/requirements-stormscope-eu.txt` handles them:

- **NATTEN** is sdist-only on PyPI, and its `setup.py` compiles the `libnatten` CUDA
  extension *only* if a GPU or `NATTEN_CUDA_ARCH` is visible at build time. Coiled builds
  software environments on CPU machines, so a PyPI install would silently produce a NATTEN
  without CUDA kernels. We pull a prebuilt wheel instead:
  `--find-links https://whl.natten.org/` + `natten==0.21.6+torch2100cu128`, matched to
  `torch==2.10.0` (whose default PyPI wheel is cu128) and Python 3.12.
- **`earth2studio[stormscope]` depends on `earth2studio[utils]`, which pulls `earth2grid`** —
  git-only, not on PyPI, and a source build. `stormscope_meteosat.py` never imports it, so we
  install the rest of the extra's contents directly (`nvidia-physicsnemo`, `nvtx`, `omegaconf`,
  `scipy`) and skip it.

`earth2studio` comes from git `main`: `StormScopeMeteosatEU` and the `MeteosatFCI` data
source are not in the 0.17.0 PyPI release. `arraylake` and `icechunk` arrive with the
`[data]` extra, so no extra pins are needed for the ingest step.

In [ ]:
import torch

import natten

print("torch     ", torch.__version__)
print("cuda avail", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device    ", torch.cuda.get_device_name(0))
print("natten    ", natten.__version__)  # 0.21.6 (installed as 0.21.6+torch2100cu128)
print("libnatten ", natten.HAS_LIBNATTEN)  # must be True, else the CUDA kernels are missing

## Set up

`StormScopeMeteosatEU` is a pure-observation nowcasting model: no NWP conditioning, the only
input is a sliding window of six consecutive FCI frames ending at the analysis time. It
operates on a fixed EU sub-region of the full MTG disk, given in native FCI pixel coordinates
by `StormScopeMeteosatEU.Model_FCI_BBox`.

In [ ]:
import os
from collections import OrderedDict
from datetime import datetime, timedelta, timezone

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()

import arraylake
import eumdac
import numpy as np
import pyproj
import torch
import xarray as xr
import zarr
from tqdm import tqdm
from zarr.codecs import BloscCodec, ZstdCodec

import earth2studio
from earth2studio.data import fetch_data
from earth2studio.data.meteosat_fci import (
    PERSPECTIVE_POINT_HEIGHT,
    SEMI_MAJOR_AXIS,
    SEMI_MINOR_AXIS,
    MeteosatFCI,
)
from earth2studio.data.utils import radiance_to_bt
from earth2studio.data.utils_eumetsat import eumetsat_credentials
from earth2studio.io import IceChunkBackend
from earth2studio.models.px.stormscope_meteosat import StormScopeMeteosatEU
from earth2studio.utils.coords import map_coords

## Load model

`load_default_package()` downloads the checkpoint from HuggingFace on first run and caches it.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

package = StormScopeMeteosatEU.load_default_package()
model = StormScopeMeteosatEU.load_model(package=package)
model = model.to(device)
model.eval()
model.compile_model()

## Set up data sources

`MeteosatFCI` is configured with `pixel_bbox` matching the model's domain (`Model_FCI_BBox`)
so the returned `y`/`x` scan-angle coordinates align with what the model expects — no
interpolation needed.

EUMETSAT serves FCI visible and NIR channels at 1 km and IR channels at 2 km. StormScope uses
the 2 km grid for all 16 channels, so we pull from both resolutions and downsample the VIS/NIR
ones. The 1 km grid is exactly 2x the pixels of the 2 km grid, so the 1 km bounding box is the
2 km one doubled.

In [ ]:
bbox_2km = StormScopeMeteosatEU.Model_FCI_BBox
FCI_BBox = {"2km": bbox_2km}
FCI_BBox["1km"] = (
    (2 * bbox_2km[0][0], 2 * bbox_2km[0][1]),
    (2 * bbox_2km[1][0], 2 * bbox_2km[1][1]),
)
fci = {
    res: MeteosatFCI(resolution=res, pixel_bbox=bbox)
    for (res, bbox) in FCI_BBox.items()
}

## Fetch the initial condition

Six consecutive FCI frames ending at the analysis time. `fetch_data` calls the data source at
`start_time + lead_time` for each offset the model defines (`[-50 min, -40 min, ..., 0 min]`).

By default the run is **as close to real time as the EUMETSAT Data Store lets this account
get**: the newest 10-minute slot that is both fully published and downloadable. Publication
is fast (~3 minutes after a scan ends), but downloading a product less than an hour past its
sensing end needs the EUMETSAT **NRT licence** — without it the fetch fails with
`403 ... NRTLicense required to access this collection`. `MIN_LAG` below keeps the search
clear of that window, putting the analysis time ~80–90 minutes behind now; set it to 0 once
the account holds the NRT licence. When the Data Store cannot be searched at all,
`start_time` falls back to the slot two hours back, which is late enough to exist under any
plausible publication delay.

**This notebook pins `START_TIME` to today 09:00 UTC** (the 6-frame input window is then
08:10–09:00). Without the NRT licence that slot is downloadable from ~10:20 UTC on. Set
`START_TIME = None` to run the freshest slot instead, or any other time to pin a case.
StormScopeMeteosatEU was trained on November 2024 – May 2026 data, so a time outside that window is a genuine out-of-sample test, inside it
a quick smoke test.

In [ ]:
# Pin a case study here -- any 10-minute FCI slot -- or set None to run the freshest
# initial condition the Data Store can serve. Default: today 09:00 UTC.
START_TIME: datetime | None = datetime.now(timezone.utc).replace(
    hour=9, minute=0, second=0, microsecond=0
)

SCAN_STEP = timedelta(seconds=MeteosatFCI.SCAN_FREQUENCY)  # FCI repeat cycle: 10 min
FCI_EPOCH = datetime(2000, 1, 1, tzinfo=timezone.utc)  # anchor of the scan-slot grid
SEARCH_LOOKBACK = timedelta(hours=6)  # how far back to hunt for a complete input window
REALTIME_LAG = timedelta(hours=2)  # blind fallback when the Data Store cannot be searched

# Youngest slot this account is allowed to *download*. The Data Store publishes a product
# ~3 minutes after its scan ends and lists it immediately, but downloading one less than an
# hour past its sensing end requires the EUMETSAT **NRT licence** on the account. Without it
# the search still returns the slot and `fetch_data` then fails with
#
#     ProductError: ... - Unauthorised (403)
#     {"exceptionText": "NRTLicense required to access this collection"}
#
# Measured against this account on 2026-09-14: slot age 65 min -> 403, 70 min -> 200. The
# 80 min below is that boundary plus a margin. Hold the NRT licence? Set it to 0 and the run
# moves to ~20 minutes behind real time.
MIN_LAG = timedelta(minutes=80)


def floor_to_slot(when: datetime) -> datetime:
    """Round `when` down onto the FCI scan grid.

    `MeteosatFCI` rejects any time that is not an exact multiple of `SCAN_FREQUENCY`
    seconds since 2000-01-01, so every candidate analysis time has to land on it.
    """
    when = (
        when.astimezone(timezone.utc)
        if when.tzinfo
        else when.replace(tzinfo=timezone.utc)
    )
    slots = int((when - FCI_EPOCH).total_seconds()) // MeteosatFCI.SCAN_FREQUENCY
    return FCI_EPOCH + slots * SCAN_STEP


def latest_fci_slot(
    n_frames: int,
    lookback: timedelta = SEARCH_LOOKBACK,
    min_lag: timedelta = MIN_LAG,
) -> datetime:
    """Newest downloadable slot whose `n_frames`-frame input window is complete.

    Slots younger than `min_lag` are excluded from the search, so the answer is the
    freshest initial condition this account can actually fetch rather than the freshest
    one the Data Store lists -- and a gap in the feed just moves it one slot further back
    instead of failing halfway through `fetch_data`.

    Searching FDHSI alone is enough: every channel the model reads comes from that
    collection, the IR/WV bands at 2 km and the VIS/NIR bands at 1 km. HRFI only repeats
    four of them at finer resolution, which this notebook does not use.
    """
    now = datetime.now(timezone.utc)
    token = eumdac.AccessToken(credentials=eumetsat_credentials())
    collection = eumdac.DataStore(token).get_collection(
        MeteosatFCI.COLLECTION_ID["FDHSI"]
    )
    products = collection.search(dtstart=now - lookback, dtend=now - min_lag)

    slots = {floor_to_slot(p.sensing_start) for p in products}
    for slot in sorted(slots, reverse=True):
        if slots.issuperset(slot - i * SCAN_STEP for i in range(n_frames)):
            return slot
    raise FileNotFoundError(
        f"no {n_frames} consecutive FDHSI slots published between {lookback} and "
        f"{min_lag} ago ({len(slots)} slots returned)"
    )


def resolve_start_time(n_frames: int) -> datetime:
    """The analysis time to run: pinned, else latest available, else `REALTIME_LAG` back."""
    if START_TIME is not None:
        return floor_to_slot(START_TIME)
    try:
        return latest_fci_slot(n_frames)
    except Exception as err:  # no credentials, no network, feed gap beyond the lookback
        fallback = floor_to_slot(datetime.now(timezone.utc) - REALTIME_LAG)
        print(f"Data Store search failed ({err!r}); falling back to {REALTIME_LAG} back")
        return fallback


In [ ]:
in_coords = model.input_coords()
variables = in_coords["variable"]

analysis_time = resolve_start_time(n_frames=len(in_coords["lead_time"]))
lag_min = (datetime.now(timezone.utc) - analysis_time).total_seconds() / 60
print(f"analysis time {analysis_time:%Y-%m-%d %H:%M} UTC ({lag_min:.0f} min behind now)")

# numpy carries no timezone; the naive stamp is UTC.
start_time = np.datetime64(analysis_time.replace(tzinfo=None))

with torch.no_grad():
    x_res = {}
    for res in ["2km", "1km"]:
        available_vars = fci[res].available_variables()
        if res == "1km":
            # use 2km version if variable available in both 1km and 2km
            available_vars -= fci["2km"].available_variables()
        variables_res = [var for var in variables if var in available_vars]
        x_res[res] = fetch_data(
            fci[res],
            time=np.array([start_time]),
            variable=variables_res,
            lead_time=in_coords["lead_time"],
            device=device,
        )

    # 2x downsample 1km data to common 2km grid
    x, coords = StormScopeMeteosatEU.combine_1km_2km_inputs(
        *x_res["1km"], *x_res["2km"]
    )
    del x_res
    # ensure data is on model grid (reorders variables if needed)
    x, coords = map_coords(x, coords, in_coords)

print(x.shape, {k: np.shape(v) for k, v in coords.items()})

## Add the ensemble dimension

The model expects `(ensemble, time, lead_time, variable, y, x)`. Up to GPU memory limits,
`ensemble_size` can be raised to produce multiple independent members in a single forward pass.

In [ ]:
ensemble_size = 1
x = x.expand(ensemble_size, -1, -1, -1, -1, -1)
coords["ensemble"] = np.arange(ensemble_size)
coords.move_to_end("ensemble", last=False)

## Ingest target: Arraylake + Icechunk

The rollout below writes to the Icechunk repo
`gtc-berlin/stormscope-eu_gtc_berlin_2026`
([web app](https://app.earthmover.io/gtc-berlin/stormscope-eu_gtc_berlin_2026)),
one commit per 10-minute step. Writing inside the loop rather than buffering the whole
rollout is the point of the near-real-time story — and it keeps peak host memory at one
frame instead of ~3.3 GB for the full cube.

**Through Earth2Studio's IO backend, not `to_icechunk`.** Earth2Studio ships
[`IceChunkBackend`](https://nvidia.github.io/earth2studio/main/modules/generated/io/earth2studio.io.IceChunkBackend.html),
an `IOBackend` that writes tensors straight into an Icechunk session's Zarr store —
`add_array` once, `write` per step, `commit` when a step is done. It is used as is, exactly
as in `06_stormscope_eu_pipeline.py`: `arraylake.Client.get_icechunk_storage(name)` returns
a plain `icechunk.Storage` (bucket and refreshed credentials resolved by Arraylake), which is
what `IceChunkBackend`'s `storage` parameter takes. The only thing the base class lacks —
writing a non-dimension 2-D coordinate (`lat`/`lon`) and the CF grid mapping scalar — is
done by two plain functions against `backend.root`, `add_coord` and `add_grid_mapping`.
The one subclass, `ReopenableIceChunkBackend`, only lets the backend reopen a repo that
holds that scalar: base `IceChunkBackend` fails on it with a `TypeError`, and appending to
an existing cube means reopening it on every run.
`add_array` has no `attrs` parameter, so array attributes are set with `.attrs.update()`.

What changes versus the `icechunk.xarray.to_icechunk` path it replaces:

- No per-step `xarray.Dataset`, no `encoding` dict, no `mode=`/`append_dim` bookkeeping.
  Chunking and compression are backend constructor arguments (`chunks`, `zarr_codecs`).
- **Each run allocates its `time` slice at full `lead_time` length up front** and each
  step writes its own `lead_time` slice, instead of that axis growing by append. Unwritten lead times read
  back as NaN (`fill_value=nan`), so the schema a consumer sees is stable from the first
  commit — which is what a service sitting in front of a live rollout wants.
- Writes are handed to a background thread pool, so the store write for step *k*
  overlaps the compute of step *k+1*; `commit` flushes them first.

**Schema.** The same as `06_stormscope_eu_pipeline.py`, written to the repo's **root
group**: two display-ready RGB composites, both `(ensemble, time, lead_time, y, x, rgb)`
float32 in 0–1. The 16 raw FCI channels are model input and output but are not stored.

- `true_color`: the EUMETSAT true-colour recipe on the three VIS channels
  (`fci06vis`/`fci05vis`/`fci04vis`).
- `airmass`: the [EUMETSAT airmass recipe](https://eumetrain.org/sites/default/files/2020-05/RGB_recipes.pdf)
  (p. 4) on WV 6.3/7.3 and IR 9.7/10.5 µm, after converting their radiances to brightness
  temperatures with `radiance_to_bt`: red WV6.3 − WV7.3, green IR9.7 − IR10.5, blue WV6.3
  inverted (dry upper levels show blue).

Storing composites means anything downstream — `.plot.imshow()`, a tile request, a
dashboard — renders them without re-deriving anything.

Coordinates, with CF standard names:

- `time` (`forecast_reference_time`) is the analysis (initialisation) time — one entry per
  model run.
- `lead_time` (`forecast_period`) is the offset from it, and **only the forecast is
  stored**: `0 min` is the observed analysis frame, `+10 … +N*10 min` the predictions. The
  generator's step 0 yields the full initial condition — six *observed* FCI frames at
  `-50 … 0 min` — but the five earlier ones are model *input*, so `forecast_only` drops
  them and the axis never goes negative.
- `valid_time` (`time`) is the 2-D `time + lead_time`.
- `ensemble` (`realization`).
- `y`/`x` are MTG scan angles in radians, with CF `projection_y/x_coordinate` attributes.
  **`x` is stored positive east**, the CF convention. earth2studio's FCI `x` is positive
  west, so `east_positive` negates it, both in the schema and in every write.

**Georeference: a CF geostationary grid mapping**, `fci_geos_projection`, which both
composites name in their `grid_mapping` attribute, plus the same CRS as GeoZarr
[`proj` and `spatial`](https://geozarr.org/conventions.html) attributes (WKT2, PROJJSON and
the affine transform) on `true_color`, `airmass`, `lat` and `lon` for readers that do not
follow CF grid mappings. That is what the
[Flux tiles service](https://docs.earthmover.io/compute/tiles) projects with. The 2-D
`lat`/`lon` (`model.lat`, `model.lon`) are kept as coordinates for xarray users, and a CF
`coordinates` attribute re-attaches them (and `valid_time`) on `xr.open_zarr`. But they
must not be the only georeference: the domain's top corners are off the Earth's disk, so
their lat/lon are NaN, and Flux drops every grid row containing one. That is everything
north of ~50°N.

**Chunking.** One chunk per `(ensemble, time, lead_time)` and 560 × 1216 pixels (`rgb`
unsplit, 2 × 2 chunks per frame, ~8 MB each), zstd-compressed. `lat`/`lon` are float32 on
the same y/x chunks, compressed with Blosc byte-shuffle + zstd, which suits smooth float
fields much better.

**Runs accumulate.** The backend opens the repo as is, never wiping it. The first run into
an empty repo creates the cube (`create_schema`); every later run appends its init to
`time` (`extend_schema`), growing `time`, `valid_time` and both composites together, in a
schema commit of its own. An init already in `time` is a rerun: it keeps its slice, which
the steps overwrite in place. All runs must share `n_steps`, since `lead_time` is fixed by
the first. Killing a run mid-rollout is safe but partial: the repo keeps whatever steps had
committed, with the remaining lead times still NaN.

**Commit metadata.** Every commit carries `time` (the init), `lead_time` (the minutes it
covers) and `status` (`schema_extended` or `ingested`), so `repo.ancestry()` can be
filtered without reading any data.

In [ ]:
ARRAYLAKE_REPO = "gtc-berlin/stormscope-eu_gtc_berlin_2026"  # the cube goes in its root group


def add_coord(
    backend: IceChunkBackend,
    name: str,
    values,
    dims: tuple[str, ...],
    attrs: dict | None = None,
    dtype=None,
    compressors=None,
) -> None:
    """Write a non-dimension coordinate, e.g. 2-D `lat`/`lon` over `(y, x)`.

    `IceChunkBackend.add_array` cannot: an n-dimensional entry in its
    CoordSystem is taken to *be* the grid and gets replaced by enumerated
    index dimensions, which would detach the array from the model's own
    `y`/`x` scan angles. It only ever touches `backend.root` (a zarr Group),
    so it is a plain function rather than a backend subclass.
    """
    values = np.asarray(values.detach().cpu() if torch.is_tensor(values) else values, dtype=dtype)
    # y/x split like the data arrays; any other dim stays whole
    chunks = tuple(
        CHUNKS[dim] if dim in ("y", "x") else size for dim, size in zip(dims, values.shape)
    )
    if name not in backend.root:
        array = backend.root.create_array(
            name,
            shape=values.shape,
            chunks=chunks,
            dtype=values.dtype,
            dimension_names=list(dims),
            compressors=compressors or backend.zarr_codecs,
        )
        array[:] = values
    if attrs:
        backend.root[name].attrs.update(attrs)


def add_grid_mapping(backend: IceChunkBackend, name: str, attrs: dict) -> None:
    """Write a CF grid mapping variable: a scalar whose attributes describe the CRS.

    Data arrays name it in their `grid_mapping` attribute. Its value is never read,
    so it is a single int32.
    """
    if name not in backend.root:
        array = backend.root.create_array(
            name, shape=(), dtype="int32", fill_value=0, dimension_names=()
        )
        array[...] = 0
    backend.root[name].attrs.update(attrs)


def geozarr_attrs(y, x) -> dict:
    """GeoZarr `proj` + `spatial` attributes for an array over this `(y, x)` grid.

    `y`/`x` are the stored (east-positive) scan angles of the pixel centres. The CRS
    is in metres, scan angle times `perspective_point_height`, so the affine transform
    is too. Its origin is the corner of pixel (0, 0); `y` runs south to north here, so
    the transform is south-up (`e > 0`).
    """
    y_m = np.asarray(y, dtype="float64") * PERSPECTIVE_POINT_HEIGHT
    x_m = np.asarray(x, dtype="float64") * PERSPECTIVE_POINT_HEIGHT
    dy, dx = y_m[1] - y_m[0], x_m[1] - x_m[0]
    if not (np.allclose(np.diff(y_m), dy) and np.allclose(np.diff(x_m), dx)):
        raise ValueError("y/x are not evenly spaced; no affine transform describes them")
    x0, y0 = x_m[0] - dx / 2, y_m[0] - dy / 2
    x1, y1 = x0 + dx * len(x_m), y0 + dy * len(y_m)
    return {
        "zarr_conventions": ZARR_CONVENTIONS,
        "proj:wkt2": FCI_CRS.to_wkt(),
        "proj:projjson": FCI_CRS.to_json_dict(),
        "spatial:dimensions": ["y", "x"],
        "spatial:shape": [len(y_m), len(x_m)],
        "spatial:transform": [float(dx), 0.0, float(x0), 0.0, float(dy), float(y0)],
        "spatial:bbox": [float(min(x0, x1)), float(min(y0, y1)),
                         float(max(x0, x1)), float(max(y0, y1))],
        "spatial:registration": "pixel",
    }

In [ ]:
# 560 x 1216 spatial chunks: 2 x 2 = 4 per frame of the default Model_FCI_BBox
# (y=1120, x=2432), ~8 MB per composite chunk instead of the whole frame (~33 MB). `rgb`
# is absent, and the backend leaves a dimension it has no chunk for unsplit, so a chunk
# holds all three bands. `lat`/`lon` use the same y/x chunks (see add_coord).
CHUNKS = {"ensemble": 1, "time": 1, "lead_time": 1, "y": 560, "x": 1216}
COMPRESSOR = ZstdCodec(level=3)
# lat/lon only: Blosc byte-shuffle groups each value's bytes (exponents together,
# mantissas together) before zstd, which compresses smooth float fields much better.
LATLON_COMPRESSOR = BloscCodec(cname="zstd", clevel=3, shuffle="shuffle")

RGB_CHANNELS = ["fci06vis", "fci05vis", "fci04vis"]  # VIS 0.6 / 0.5 / 0.4 um
RGB_VALUES = np.array(["red", "green", "blue"])

# FCI central wavelengths; `radiance_to_bt` takes their reciprocal, the wavenumber in cm-1.
WAVELENGTH_CM = {
    "fci04vis": 0.444e-4,
    "fci05vis": 0.510e-4,
    "fci06vis": 0.640e-4,
    "fci08vis": 0.865e-4,
    "fci09vis": 0.914e-4,
    "fci13nir": 1.380e-4,
    "fci16nir": 1.610e-4,
    "fci22nir": 2.250e-4,
    "fci38ir": 3.800e-4,
    "fci63wv": 6.300e-4,
    "fci73wv": 7.350e-4,
    "fci87ir": 8.700e-4,
    "fci97ir": 9.660e-4,
    "fci105ir": 10.500e-4,
    "fci123ir": 12.300e-4,
    "fci133ir": 13.300e-4,
}
AIRMASS_CHANNELS = ["fci63wv", "fci73wv", "fci97ir", "fci105ir"]

# MTG-I1 FCI fixed grid as a CF geostationary projection. `x`/`y` are scan angles in
# radians; `pyproj.CRS.from_cf` scales them by `perspective_point_height`. Checked against
# `model.lat`/`model.lon`: max error 4e-6 deg, identical off-disk mask.
GRID_MAPPING = "fci_geos_projection"
FCI_GEOS = {
    "long_name": "MTG-I1 FCI fixed grid projection",
    "grid_mapping_name": "geostationary",
    "perspective_point_height": PERSPECTIVE_POINT_HEIGHT,
    "semi_major_axis": SEMI_MAJOR_AXIS,
    "semi_minor_axis": SEMI_MINOR_AXIS,
    "inverse_flattening": 298.257223563,
    "latitude_of_projection_origin": 0.0,
    "longitude_of_projection_origin": 0.0,
    # MTG and SEVIRI sweep about "y", GOES about "x". The wrong axis shifts positions
    # by up to 0.35 deg, with no error raised.
    "sweep_angle_axis": "y",
}
FCI_CRS = pyproj.CRS.from_cf(FCI_GEOS)
FCI_GEOS["crs_wkt"] = FCI_CRS.to_wkt()  # for GDAL/rioxarray

# GeoZarr `proj` and `spatial` conventions (https://geozarr.org/conventions.html), the
# same CRS for readers that do not follow CF grid mappings. See geozarr_attrs.
ZARR_CONVENTIONS = [
    {
        "schema_url": "https://raw.githubusercontent.com/zarr-conventions/proj/refs/tags/v0.1/schema.json",
        "spec_url": "https://github.com/zarr-conventions/proj/blob/v0.1/README.md",
        "uuid": "f17cb550-5864-4468-aeb7-f3180cfb622f",
        "name": "proj",
        "description": "Coordinate reference system information for geospatial data",
    },
    {
        "schema_url": "https://raw.githubusercontent.com/zarr-conventions/spatial/refs/tags/v0.1/schema.json",
        "spec_url": "https://github.com/zarr-conventions/spatial/blob/v0.1/README.md",
        "uuid": "689b58e2-cf7b-45e0-9fff-9cfc0883d6b4",
        "name": "spatial",
        "description": "Spatial coordinate information",
    },
]

# `grid_mapping` is what the Flux tiles service projects with. `coordinates` points at the
# non-dimension coordinates written by `add_coord`; without it `xr.open_zarr` would read
# them back as data variables instead of coordinates of every field.
GEOREF = {"coordinates": "lat lon valid_time", "grid_mapping": GRID_MAPPING}

In [ ]:
def true_color(arr, variables):
    """EUMETSAT-style "true-colour" RGB composite from the three VIS channels.

    Takes the (..., variable, y, x) array of a rollout step and returns
    (..., y, x, rgb) floats in 0..1 -- display-ready, so plotting and tiling
    need no per-channel scaling.
    """
    idx = [list(variables).index(ch) for ch in RGB_CHANNELS]
    rgb = np.clip(arr[..., idx, :, :] / 24.0, 0, 1) * 255
    rgb = np.interp(
        rgb, [0, 30, 60, 120, 190, 255], [0, 110, 160, 210, 240, 255]
    ) / 255.0
    return np.moveaxis(rgb, -3, -1).astype("float32")


def airmass(arr, variables):
    """EUMETSAT airmass RGB composite from the WV 6.3/7.3 and IR 9.7/10.5 um channels.

    Same (..., variable, y, x) -> (..., y, x, rgb) 0..1 contract as `true_color`. The
    radiances are converted to brightness temperatures (K) first. Off-disk pixels stay
    NaN.

    EUMETSAT recipe (EUMeTrain, "Compilation of RGB Recipes", p. 4): red WV6.2 - WV7.3,
    green IR9.7 - IR10.8, blue WV6.2 inverted, gamma 1 -- on FCI, WV6.3 and IR10.5. The
    ranges here are within ~1.5 K of the recipe's (-25..0, -40..+5, 243..208 K).
    """
    bt = {
        var: radiance_to_bt(arr[..., list(variables).index(var), :, :], 1.0 / WAVELENGTH_CM[var])
        for var in AIRMASS_CHANNELS
    }
    r = ((bt["fci63wv"] - bt["fci73wv"]) - (-23.8)) / (1.4 - (-23.8))
    g = ((bt["fci97ir"] - bt["fci105ir"]) - (-39.7)) / (4.1 - (-39.7))
    b = (bt["fci63wv"] - 244.5) / (209.4 - 244.5)  # WV6.3 inverted: dry upper levels -> blue
    return np.clip(np.stack([r, g, b], axis=-1), 0, 1).astype("float32")


def forecast_only(x_pred, coords_pred):
    """Trim a rollout step to `lead_time >= 0` -- the forecast, analysis frame included.

    The generator's step 0 yields the initial condition: the six *observed* FCI frames
    at -50..0 min. Only the last of them, lead 0, is part of the forecast; the five
    earlier ones are model input, so they are dropped and never reach Icechunk.
    Steps 1..N are already all-positive and pass through untouched.
    """
    lead = coords_pred["lead_time"]
    start = int(np.searchsorted(lead, np.timedelta64(0, "s")))
    if start == 0:
        return x_pred, coords_pred
    coords_out = coords_pred.copy()
    coords_out["lead_time"] = lead[start:]
    return x_pred[:, :, start:], coords_out


def east_positive(coords):
    """`coords` with `x` negated to CF's east-positive scan angle.

    earth2studio's FCI `x` is positive west. Stored as is, every CF reader (pyproj, GDAL,
    Flux) would mirror the image east-west. Applied to the schema *and* to every write,
    because `write` locates a slice by looking up coordinate values in the stored ones.
    """
    out = OrderedDict(coords)
    out["x"] = -np.asarray(out["x"])
    return out


def step_writes(x_pred, coords_pred, channels):
    """One rollout step -> the `(tensors, coords, names)` to hand `write`.

    Only the two composites are stored, not the raw channels. Both are `(ensemble,
    time, lead_time, y, x, rgb)`, so one `write` call covers them.

    `copy=True` because the generator keeps writing into its own buffers between
    yields -- and with non-blocking writes, the store write for this step may still be
    reading the array after the loop has moved on.
    """
    arr = x_pred.detach().to("cpu", torch.float32, copy=True).numpy()

    rgb_coords_step = east_positive(
        (dim, values) for dim, values in coords_pred.items() if dim != "variable"
    )
    rgb_coords_step["rgb"] = RGB_VALUES

    return (
        [torch.from_numpy(true_color(arr, channels)), torch.from_numpy(airmass(arr, channels))],
        rgb_coords_step,
        ["true_color", "airmass"],
    )


def lead_minutes(lead_time):
    return (lead_time / np.timedelta64(1, "m")).astype(int)


class ReopenableIceChunkBackend(IceChunkBackend):
    """`IceChunkBackend` that can reopen a repo holding a scalar array.

    Runs append to the existing cube, so the repo is reopened on every run. Base
    `_read_store_state` iterates each array's `dimension_names`, and zarr stores a
    scalar's `()` as None -- the CF grid mapping (see add_grid_mapping) is a scalar,
    so opening fails with a TypeError. Scalars have no dims or chunks to record, so
    this version of the scan just skips them.
    """

    def _read_store_state(self, chunks: dict[str, int]) -> None:
        self.coords = OrderedDict()
        self.chunks = chunks.copy()
        arrays = [(name, array) for name, array in self.root.arrays() if array.ndim > 0]
        for _, array in arrays:
            for dim in array.metadata.dimension_names:
                if dim not in self.coords:
                    self.coords[dim] = self.root[dim][:]
        for name, array in arrays:
            if name not in self.coords:
                self.chunks.update(zip(array.metadata.dimension_names, array.chunks))


def create_schema(
    backend: IceChunkBackend, model: StormScopeMeteosatEU, coords: OrderedDict, lead_time: np.ndarray
) -> None:
    """Allocate the cube in an empty repo, sized for this run's init alone.

    Only the forecast is stored, so `lead_time` runs from 0 -- the observed analysis
    frame -- to n_steps * 10 min, and each rollout step fills its own slice. Later
    runs grow `time` by their own init (see `extend_schema`), so `time` only ever
    lists inits that were actually generated.
    """
    total_coords = east_positive(
        OrderedDict(
            ensemble=coords["ensemble"],
            time=coords["time"],
            lead_time=lead_time,
            y=coords["y"],
            x=coords["x"],
        )
    )
    rgb_coords = OrderedDict(**total_coords, rgb=RGB_VALUES)

    backend.add_array(total_coords, [])  # coordinate arrays only
    backend.add_array(rgb_coords, ["true_color", "airmass"], fill_value=float("nan"))
    backend.root["true_color"].attrs.update(
        {
            **GEOREF,
            "long_name": "true-colour RGB composite (VIS 0.6/0.5/0.4 um), 0-1",
            "source_channels": " ".join(RGB_CHANNELS),
        }
    )
    backend.root["airmass"].attrs.update(
        {
            **GEOREF,
            "long_name": "EUMETSAT airmass RGB composite (WV 6.3-7.3, IR 9.7-10.5, "
                         "WV 6.3 um inverted brightness temperatures), 0-1",
            "source_channels": " ".join(AIRMASS_CHANNELS),
        }
    )
    add_coord(
        backend, "lat", model.lat, ("y", "x"),
        {"standard_name": "latitude", "units": "degrees_north"},
        dtype="float32", compressors=[LATLON_COMPRESSOR],
    )
    add_coord(
        backend, "lon", model.lon, ("y", "x"),
        {"standard_name": "longitude", "units": "degrees_east"},
        dtype="float32", compressors=[LATLON_COMPRESSOR],
    )
    add_grid_mapping(backend, GRID_MAPPING, FCI_GEOS)
    geozarr = geozarr_attrs(total_coords["y"], total_coords["x"])
    for name in ("true_color", "airmass", "lat", "lon"):
        backend.root[name].attrs.update(geozarr)
    for dim, axis, direction in (("x", "X", "east"), ("y", "Y", "north")):
        backend.root[dim].attrs.update(
            {
                "standard_name": f"projection_{dim}_coordinate",
                "units": "rad",
                "axis": axis,
                "long_name": f"FCI fixed grid projection {dim}-coordinate "
                             f"(scan angle, positive {direction})",
            }
        )
    # valid_time is 2-D: one run per `time`, each with its own `lead_time` offsets.
    add_coord(
        backend, "valid_time",
        total_coords["time"][:, None] + total_coords["lead_time"][None, :],
        ("time", "lead_time"),
        {"standard_name": "time", "long_name": "valid time (time + lead_time)"},
    )
    backend.root["time"].attrs.update(
        {
            "standard_name": "forecast_reference_time",
            "long_name": "analysis (nowcast initialisation) time",
        }
    )
    backend.root["lead_time"].attrs.update(
        {
            "standard_name": "forecast_period",
            "long_name": "offset from analysis time; 0 = observed analysis frame, > 0 = "
                         "forecast. Earlier observed frames are model input and are not stored.",
        }
    )
    backend.root["ensemble"].attrs.update(
        {"standard_name": "realization", "long_name": "ensemble member"}
    )
    backend.root.attrs.update(
        {
            "title": "StormScope-EU nowcast on MTG-I1 FCI",
            "model": "StormScopeMeteosatEU",
            "earth2studio_version": earth2studio.__version__,
            "fci_pixel_bbox": str(StormScopeMeteosatEU.Model_FCI_BBox),
        }
    )


# Every array with a `time` dimension; `extend_schema` grows them together.
TIME_ARRAYS = ("time", "valid_time", "true_color", "airmass")


def extend_schema(backend: IceChunkBackend, time: np.ndarray, lead_time: np.ndarray) -> str | None:
    """Append this run's init to `time` in an existing cube. Returns the commit message.

    The new `time` slice of every data array reads as NaN (fill value) until the
    rollout steps write it. An init already in `time` is a rerun: it keeps its slice,
    which the steps overwrite in place, and there is nothing to commit (returns None).
    An init older than the newest one is still appended at the end, so `time` is in
    commit order, not necessarily sorted.
    """
    stored_lead = backend.root["lead_time"][:]
    if not np.array_equal(stored_lead, lead_time):
        raise ValueError(
            f"repo has {len(stored_lead)} lead times (--n-steps {len(stored_lead) - 1}), this "
            f"run has {len(lead_time)} (--n-steps {len(lead_time) - 1}); rerun with the repo's --n-steps"
        )
    init = np.datetime_as_string(time[0], unit="m")
    stored_time = backend.coords["time"]
    if np.isin(time, stored_time).all():
        print(f"init {init} already in repo; overwriting its slice")
        return None

    n = len(stored_time)
    for name in TIME_ARRAYS:
        array = backend.root[name]
        shape = list(array.shape)
        shape[array.metadata.dimension_names.index("time")] += len(time)
        array.resize(tuple(shape))
    backend.root["time"][n:] = time
    backend.root["valid_time"][n:] = time[:, None] + lead_time[None, :]
    # `write` locates slices through this in-memory copy, which is read once at open.
    backend.coords["time"] = np.concatenate([stored_time, time])
    return f"stormscope-eu {init} extend schema"

In [ ]:
n_steps = 12  # 2 hours of 10-minute forecast steps; must match the repo's

channels = [str(v) for v in coords["variable"]]

# get_icechunk_storage returns a plain icechunk.Storage -- exactly what IceChunkBackend's
# `storage` param accepts, so no Arraylake-specific subclass is needed. Opened as is, so
# earlier runs are kept.
client = arraylake.Client(token=os.environ["ARRAYLAKE_API_TOKEN"])
io_backend = ReopenableIceChunkBackend(
    storage=client.get_icechunk_storage(ARRAYLAKE_REPO),
    chunks=CHUNKS,
    zarr_codecs=[COMPRESSOR],
)

lead_time = np.arange(n_steps + 1) * model.output_times[0]
init = np.datetime_as_string(coords["time"][0], unit="m")
if "true_color" not in io_backend.root:
    create_schema(io_backend, model, coords, lead_time)
    message = f"stormscope-eu {init} schema"
else:
    message = extend_schema(io_backend, coords["time"], lead_time)
if message is not None:
    # Commit metadata makes `repo.ancestry()` filterable without reading data:
    # `time` is the init, `lead_time` the minutes the commit covers.
    metadata = {
        "time": init,
        "lead_time": lead_minutes(lead_time).tolist(),
        "status": "schema_extended",
    }
    print("schema ->", io_backend.commit(message, metadata=metadata))
print(sorted(io_backend.root.array_keys()))

## Execute the nowcast, streaming to Icechunk

`create_iterator` handles the autoregressive rollout: the first yielded value is the
(unchanged) initial condition, each subsequent one advances the prediction by 10 minutes.
Every step is written into its `lead_time` slice of the arrays allocated above and
committed before the next one is computed.

**Only the forecast is ingested.** `forecast_only` trims each step to `lead_time >= 0`, so
step 0 contributes just its analysis frame (lead 0) and the five earlier observed frames
that the model consumed as input are dropped. The cube therefore runs `0 … +N*10 min` with
no negative lead times.

In [ ]:
for step, (x_pred, coords_pred) in enumerate(
    tqdm(model.create_iterator(x, coords), total=n_steps + 1)
):
    # Forecast only: step 0's five past observed frames are model input, not output.
    x_pred, coords_pred = forecast_only(x_pred, coords_pred)

    io_backend.write(*step_writes(x_pred, coords_pred, channels))

    lead = lead_minutes(coords_pred["lead_time"])
    if step == 0:
        # the analysis frame: the single observed frame at lead 0
        message = f"stormscope-eu {init} analysis frame (lead {lead[-1]} min)"
    else:
        message = f"stormscope-eu {init} +{lead[-1]} min"

    # `commit` flushes the background writes above, then opens a fresh session --
    # an Icechunk session is spent once committed.
    metadata = {"time": init, "lead_time": int(lead[-1]), "status": "ingested"}
    tqdm.write(f"{message} -> {io_backend.commit(message, metadata=metadata)}")

    if step == n_steps:
        break

## Verify the round trip

Reopen the committed cube from Arraylake. `consolidated=False` is required for Icechunk
stores, and `chunks=None` skips Dask — we only want the metadata and a small slice.

In [ ]:
ds_al = xr.open_zarr(
    io_backend.repo.readonly_session("main").store,
    consolidated=False,
    chunks=None,
)

print(ds_al)
print()
print("inits:  ", ds_al["time"].values)
print("commits:", sum(1 for _ in io_backend.repo.ancestry(branch="main")))
print("web:    ", f"https://app.earthmover.io/{ARRAYLAKE_REPO}")

# `xr.open_zarr` moves `grid_mapping` into `.encoding`, so check the raw Zarr attributes:
# without them the tiles service falls back to lat/lon and loses everything north of ~50N.
stored = zarr.open_group(io_backend.repo.readonly_session("main").store, mode="r")
assert all(stored[v].attrs.get("grid_mapping") == GRID_MAPPING for v in ["true_color", "airmass"])
assert stored["x"].attrs["standard_name"] == "projection_x_coordinate"
assert all("proj:wkt2" in stored[v].attrs for v in ["true_color", "airmass", "lat", "lon"])
assert stored["lat"].dtype == np.float32
last = next(io_backend.repo.ancestry(branch="main"))
# every commit is tagged; the newest is this run's last step
assert (last.metadata.get("time"), last.metadata.get("status")) == (init, "ingested")

In [ ]:
# Both composites are stored display-ready, so plotting them is a plain slice
# of this run's init (the cube may hold earlier ones too).
composites = ds_al[["true_color", "airmass"]].isel(ensemble=0).sel(time=coords["time"][0])

In [ ]:
for name in ["true_color", "airmass"]:
    composites[name].isel(lead_time=[0, -1]).plot.imshow(col="lead_time", rgb="rgb")